<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c41-header.png" alt="Nextia Learning · Speech AI: Transcribe, Understand and Speak" width="100%">

# Spot commands on a small device

**[Spot commands on a small device](https://learning.nextia-ai.com/courses/speech/m10/spot-commands-on-a-small-device/)** · Speech AI: Transcribe, Understand and Speak · Case study 3 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** build and compare keyword spotters on real recordings: an energy-and-template baseline, a small CNN on log-mel pictures, Whisper tiny with a word list and Vosk with a grammar. You will measure commands caught and false alarms per hour, choose each threshold for a false-alarm budget on validation, test once, compare speed and size, and run the winner on a new recording.

| | |
|---|---|
| **Time** | About 2.5 hours with the page; the notebook runs in a few minutes |
| **Needs** | An internet connection for the setup cells (about 36 MB; the optional live cell downloads 115 MB more). No account, no key and no graphics card. |
| **You should know** | Keyword spotting, recall and false alarms, from [Compliance phrases and keyword spotting](https://learning.nextia-ai.com/courses/speech/m06/compliance-phrases-and-keyword-spotting/); a threshold chosen on validation, from the machine learning course. |
| **Tested** | Python 3.12 with NumPy 2.5.4, SciPy 1.18.1, soundfile 0.14.0, matplotlib 3.11.2, PyTorch 2.15.0, faster-whisper 1.2.1 and Vosk 0.3.44, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/speech/m10/spot-commands-on-a-small-device/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/M10-L03-spot-commands-on-a-small-device/spot-commands-on-a-small-device-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: NumPy 2.5.4, SciPy 1.18.1, soundfile 0.14.0, matplotlib, faster-whisper 1.2.1 and Vosk 0.3.44. It installs PyTorch 2.15.0 (CPU) only if PyTorch is missing: Colab and Kaggle already have a version of it. It needs an internet connection and takes one or two minutes the first time.

You should see one line per package with its version. With another PyTorch version, the CNN's numbers can differ a little from the page; the checks allow for that. If the versions of the other packages differ, restart the kernel (in Colab: **Runtime › Restart session**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 scipy==1.18.1 soundfile==0.14.0 matplotlib==3.11.2 faster-whisper==1.2.1 vosk==0.3.44
import importlib.util, subprocess, sys
if importlib.util.find_spec("torch") is None:
    # the CPU build of PyTorch (about 200 MB), not the larger graphics-card build
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.15.0",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ["numpy", "scipy", "soundfile", "torch", "faster-whisper", "vosk"]:
    print(name, version(name))

## Setup: get the clips

Run the next cell. It downloads the files of this case study (about 37 MB) into a folder `speech-commands`, with the course's helper `compare.py`, and checks each file's checksum. It needs an internet connection and no account.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `speech-commands`, then run the cell again.

What you get: a fixed sample of Speech Commands v0.02 — 9,185 one-second clips and 1,720 noise windows, stored as **log-mel pictures** (section 2 explains them) — with their labels; 24 test clips and one kitchen recording as audio; and the **recorded** outputs of Whisper tiny and Vosk on every validation and test item. The whole dataset is 2.4 GB, so the notebook works from the pictures and the recorded outputs, and runs both models live on a few clips.

Source: [Speech Commands v0.02](https://arxiv.org/abs/1804.03209), Pete Warden (Google), [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Nextia Learning took a fixed sample, turned the clips into log-mel pictures, cut the background-noise files into one-second windows, and placed six test clips in a kitchen recording. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/M10-L03-spot-commands-on-a-small-device/dataset.md) has the details.

In [ ]:
# Setup: download the files into speech-commands/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/"
HERE_FOLDER = "M10-L03-spot-commands-on-a-small-device/"
FILES = {
    "data/LICENSE-speech-commands.txt": "fd3a263fe19ed8faa9068b43abaebafc02c77897b0c6fc09abc04bb592e5f16e",
    "data/audio/bed__1b4c9b89_nohash_0.wav": "a772f20df84381acf708c9481e5eb3e10ce9b4df007dcfb7bcc23d67c038eaa6",
    "data/audio/down__2d82a556_nohash_1.wav": "9ae79ba54c2a692cb554b4c03af1438ac50ba81b5c4105ee815139f75dae8175",
    "data/audio/down__5e3dde6b_nohash_2.wav": "201bd47cf1d22f58efdf9f8a1ae208e8b78008152cd68d8884dc8e8f91b1f419",
    "data/audio/go__68dd409e_nohash_0.wav": "d8a79753ed07c42236c12a0727f5f81aab5170caf9ac7e002d985feb2544bc73",
    "data/audio/go__d962e5ac_nohash_3.wav": "a1d0ecdb50d3020e8d33de65315bd328860f15dfb92399669e20bfaebf181ee8",
    "data/audio/kitchen-stream.wav": "454175cbf4d9220c915cd6592bec37a01352637a62e0315cad6fafcf16cb537d",
    "data/audio/left__4c6167ca_nohash_6.wav": "fd74edf8e91bee09688e0f0285d69e48e3484646e80c318db6ee8f09d7d45b21",
    "data/audio/left__a8e25ebb_nohash_0.wav": "708e06736d95d0b1f9ae4197e913ca814a3616237e2dd09464acafb3d9260543",
    "data/audio/nine__4c6167ca_nohash_0.wav": "a44f95fa9370231a1cd1eb4b748925e6e9a773c372f929926e7a87f5ac7512ae",
    "data/audio/nine__6b889021_nohash_0.wav": "ee68c511d513daae141666113d2503c5ebcbcffd104a5240d647a4c39c3128e3",
    "data/audio/no__2796ac50_nohash_1.wav": "a6dec1ea6cf8535f7685b08a79136a9a7741ebc20f2447afa876d8ddd52d90b0",
    "data/audio/no__c7dc7278_nohash_0.wav": "c6b600b0c279d6488f2d97b72f33dd4781da76ce584fbadaf2ebbd7a9ae77bca",
    "data/audio/off__e0c782d5_nohash_1.wav": "8b832c03965c30835cb7c4c701009df7ff35efceb4b1ec2ed0a7321b70055297",
    "data/audio/off__f297e878_nohash_1.wav": "a1b8060dc379ae51ed80ede636e357715da3a611d3f80d7fa282f9461ddb48ae",
    "data/audio/on__63f7a489_nohash_0.wav": "341c3a02d076526c92a7ac7ebd1e9250a6b111d00da0643fcafa82fbd814da9e",
    "data/audio/on__e41a903b_nohash_1.wav": "11c427d1f57e1adf25f3d1e4ad8958289dc2e89c57dfd631dfce75e871461566",
    "data/audio/right__a60a09cf_nohash_0.wav": "c7372dcc2797e2aa94d0589660de944601e6718dc6ac58bf97648dab15cf7678",
    "data/audio/right__e1469561_nohash_4.wav": "e6e4cefeef75f4d1770de154b88c8299ff654941ff2dbf2d678589fe6d0ed957",
    "data/audio/stop__0c40e715_nohash_0.wav": "2e255e08ad87c074fa6d92750251e12f38df033a3335a4672354472489603560",
    "data/audio/stop__e71b4ce6_nohash_1.wav": "51385d6a7fdcc9d7d04befc660d029a1ab531f6202aaf0d9ed4ec67d7c81c332",
    "data/audio/three__8769c34c_nohash_2.wav": "d74c2c6c2ab8c0acca4d38b27097421149b8f64b97a27a6d9e72c66097565143",
    "data/audio/up__18f8afd5_nohash_1.wav": "e4d81d41ae82784fb1cd4d64a9fbd24407db50e43a3944d5de1a8769f607ddac",
    "data/audio/up__beb458a4_nohash_2.wav": "c0266f77c0d18941a8126e6a3615a271de0ba2354975763ef43a8899d499039e",
    "data/audio/yes__3df9a3d4_nohash_1.wav": "75429f5f02c04c51382b080a6565ea2c57c89648400cd6d4f7535d58cf6f6ccf",
    "data/audio/yes__43fc47a7_nohash_0.wav": "8757c6c681363f7aa69af411a5ccd3942b2f40747b73ffd759701529790a15c1",
    "data/clips.csv": "e20ce0bec2408afe072c601242d3720a21086bb646ed2ce075f1952babd146e3",
    "data/recordings/latency.json": "59209911b1fc741228ca2a7cededb446ffa873512d28ec49f50596ec0a732847",
    "data/recordings/vosk-small.jsonl": "3129cded0e2e0b4931a6b748cb11d3d734eeae88f4e5b3af67b5eae82cadcfe5",
    "data/recordings/whisper-tiny.jsonl": "07f0139c5e5e07b665f7fe9445a1871f8fb19be14ae377833ee1b43c7722d092",
    "data/speech-commands-sample.npz": "79e422e7b5d9b5100638b46d21d9ad0e6201527237d5129aa330cf5935ea7e83",
    "data/stream.json": "8786ebaf8ae172ef4e30845a63bee8c104747acd911b5c5525139e2cfc8cad97"
}
FILES["compare.py"] = "db394eb044e0423706644ba7c8e6451e4d21a268293b868032918ea2f30db85b"
HERE = Path.cwd()
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("speech-commands").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + (name if name == "compare.py" else HERE_FOLDER + name), path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder speech-commands and run again.")
os.chdir(PROJECT)
print(f"Ready: {len(FILES)} files in speech-commands.")

## Setup: helpers

The next cell loads NumPy, soundfile, matplotlib, the audio player and the course helper `compare.py` as `cp` (its normalisation guide reads Whisper's text). It defines the ten commands, the twelve classes the CNN learns, and the check helpers: `expect(what, yours, expected)` and `expect_hash(what, yours, expected)` tell you if an answer is right without showing it.

Run the cell. It prints the classes.

In [ ]:
# Helpers: the words, and checks that do not show the answer.
import csv, hashlib, json, time, warnings
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from IPython.display import Audio, display
import compare as cp
warnings.filterwarnings("ignore")

COMMANDS = ["yes", "no", "up", "down", "left", "right", "on", "off", "stop", "go"]
CLASSES = COMMANDS + ["other", "noise"]   # "other": a word that is not a command; "noise": no speech
print(CLASSES)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The situation

Larkfield's product team wants its **Mistral fan** to obey ten spoken commands: on, off, up, down, left, right, stop, go, yes and no. The fan has a small chip and no internet, so the spotter runs on the fan and no audio leaves the room.

- A **false alarm** (the fan acts when nobody gave a command) is the worst error. The budget: at most **10 false alarms per hour** of speech and noise that is not a command.
- A **missed command** is bad, but the person can say it again.

## 2. The data, briefly

Run the next cell. It loads the log-mel pictures and the labels of the sample, and counts each part.

In [ ]:
data = np.load("data/speech-commands-sample.npz")
pictures = data["pictures"]                      # one 40 x 98 picture per item, one byte a value
clips = list(csv.DictReader(open("data/clips.csv", encoding="utf-8")))
split = np.array([c["split"] for c in clips])    # train, valid or test
label = np.array([c["class"] for c in clips])    # a command, "other" or "noise"
speaker = np.array([c["speaker"] for c in clips])
y = np.array([CLASSES.index(c) for c in label])  # the class number, 0 to 11
print(pictures.shape, len(clips))
for s in ["train", "valid", "test"]:
    m = split == s
    n_cmd, n_other, n_noise = (m & (y < 10)).sum(), (m & (label == "other")).sum(), (m & (label == "noise")).sum()
    print(f"{s:5}  commands {n_cmd:5}   other words {n_other:5}   noise windows {n_noise:5}   speakers {len(set(speaker[m & (y < 11)])):5}")

> **Check.** 10905 items. Train has commands, other words and noise windows; validation and test have new speakers.

The dataset puts every clip of one speaker in the same split (a hash of the speaker's ID). Check it, and see which words count as "other".

In [ ]:
people = {s: set(speaker[(split == s) & (label != "noise")]) for s in ["train", "valid", "test"]}
shared = len(people["train"] & people["test"]) + len(people["train"] & people["valid"]) + len(people["valid"] & people["test"])
print("speakers in two splits:", shared)
print("other words in the test:", sorted({c["word"] for c in clips if c["split"] == "test" and c["class"] == "other"}))

In [ ]:
expect('speakers in two splits', shared, 0)

Listen to a command and another word. Neither player starts by itself.

In [ ]:
for name in ["yes__43fc47a7_nohash_0.wav", "bed__1b4c9b89_nohash_0.wav"]:
    x, sr = sf.read(f"data/audio/{name}")
    print(name, f"{len(x) / sr:.2f} s at {sr} samples a second")
    display(Audio(x, rate=sr))

**Transcripts.** "Yes." and "Bed." Two volunteers of Speech Commands v0.02 (CC BY 4.0).

## 3. Sound as a picture: the log-mel spectrogram

A **log-mel spectrogram** groups the pitches into 40 mel bands (narrow at low pitch, wide at high pitch, as the ear hears) and gives the loudness of each band in each 10 ms frame, in decibels. One second becomes a 40 x 98 picture. The next cell defines `log_mel`, the same code that made the stored pictures.

In [ ]:
SR = 16000          # samples a second
N_FFT, WIN, HOP = 512, 400, 160   # 25 ms windows, every 10 ms
N_MELS = 40         # mel bands from 20 to 7,600 Hz

def mel_filters(sr=SR, n_fft=N_FFT, n_mels=N_MELS, fmin=20.0, fmax=7600.0):
    """Triangle filters, evenly spaced on the mel scale (close together at low pitch)."""
    mel = lambda f: 2595 * np.log10(1 + f / 700)
    hz = lambda m: 700 * (10 ** (m / 2595) - 1)
    edges = hz(np.linspace(mel(fmin), mel(fmax), n_mels + 2))
    bins = np.fft.rfftfreq(n_fft, 1 / sr)
    fb = np.zeros((n_mels, len(bins)))
    for i in range(n_mels):
        lo, mid, hi = edges[i], edges[i + 1], edges[i + 2]
        fb[i] = np.clip(np.minimum((bins - lo) / (mid - lo), (hi - bins) / (hi - mid)), 0, None)
    return fb

FILTERS = mel_filters()

def log_mel(x):
    """One second of audio (16,000 numbers) -> a 40 x 98 picture: loudness in dB per mel band and 10 ms frame."""
    x = np.asarray(x, dtype=np.float64)[:SR]
    x = np.pad(x, (0, SR - len(x)))
    n_frames = 1 + (SR - WIN) // HOP                     # 98 frames
    idx = np.arange(WIN)[None, :] + HOP * np.arange(n_frames)[:, None]
    frames = x[idx] * np.hanning(WIN)
    power = np.abs(np.fft.rfft(frames, n=N_FFT)) ** 2     # 98 x 257
    return 10 * np.log10(FILTERS @ power.T + 1e-10)       # 40 x 98, in dB

def to_uint8(db):
    """Store a log-mel picture in one byte a value: -100 dB to +27.5 dB in steps of 0.5 dB."""
    return np.clip(np.round((db + 100) * 2), 0, 255).astype(np.uint8)

def from_uint8(q):
    return q.astype(np.float32) / 2 - 100

Compute the picture of the "yes" clip from its audio, and compare it with the stored one.

In [ ]:
x, sr = sf.read("data/audio/yes__43fc47a7_nohash_0.wav")
mine = log_mel(x)
i = [c["id"] for c in clips].index("yes/43fc47a7_nohash_0.wav")
stored = from_uint8(pictures[i])
print(mine.shape, "largest difference from the stored picture:", round(float(np.abs(np.clip(mine, -100, 27.5) - stored).max()), 2), "dB")

> **Check.** `(40, 98)` and a largest difference of about 0.25 dB: the stored pictures keep one value every 0.5 dB.

Now look at a command, another word and noise.

In [ ]:
show = {"yes": "yes/43fc47a7_nohash_0.wav", "bed (other word)": "bed/1b4c9b89_nohash_0.wav",
        "noise (running tap)": "_background_noise_/running_tap@52.9"}
ids = [c["id"] for c in clips]
fig, axes = plt.subplots(1, 3, figsize=(11, 3), sharey=True)
for ax, (title, item) in zip(axes, show.items()):
    ax.imshow(from_uint8(pictures[ids.index(item)]), origin="lower", aspect="auto", cmap="magma", vmin=-80, vmax=25)
    ax.set_title(title); ax.set_xlabel("10 ms frames")
axes[0].set_ylabel("mel band (low to high)")
plt.tight_layout(); plt.show()

> **Check.** "yes" is a shape in the middle, with a bright top at the end (the "s"). "bed" is a shorter shape. The tap is a texture with no shape.

## 4. The measure

`evaluate` turns a method's answers into the numbers the product team uses. For each item, the method gives the command it heard (or `""` for none) and a score. The device **acts** when it heard a command with a score of at least the threshold.

- **right**: the share of command clips where it acts with the right word; **wrong word** and **missed** are the rest.
- **false alarms per hour**: the times it acts on other words or noise, divided by the hours of that audio (an other-word clip is 1 s; a noise window adds 0.5 s).

In [ ]:
HOURS_PER = {"other": 1.0 / 3600, "noise": 0.5 / 3600}   # an other-word clip is 1 s; a noise window adds 0.5 s

def evaluate(heard, score, threshold, part):
    """heard[i]: the command a method heard in item i, or "" for none; score[i]: how sure it is.
    The device acts when it heard a command with a score of at least `threshold`."""
    m = split == part
    acts = (heard != "") & (score >= threshold)
    out = {"per_word": {}}
    for w in COMMANDS:
        k = m & (label == w)
        out["per_word"][w] = round(float((acts[k] & (heard[k] == w)).mean()), 3)
    k = m & (y < 10)
    out["right"] = round(float((acts[k] & (heard[k] == label[k])).mean()), 3)   # acted, with the right word
    out["wrong word"] = round(float((acts[k] & (heard[k] != label[k])).mean()), 3)
    out["missed"] = round(float((~acts[k]).mean()), 3)                           # the person must say it again
    for kind in ["other", "noise"]:
        k = m & (label == kind)
        out[f"false alarms ({kind})"] = int(acts[k].sum())
        out[f"per hour ({kind})"] = round(float(acts[k].sum() / (k.sum() * HOURS_PER[kind])), 1)
    k = m & (y >= 10)
    hours = (m & (label == "other")).sum() * HOURS_PER["other"] + (m & (label == "noise")).sum() * HOURS_PER["noise"]
    out["false alarms per hour"] = round(float(acts[k].sum() / hours), 1)
    return out

def summary(result):
    return {k: v for k, v in result.items() if k != "per_word"}

## 5. A baseline: energy and templates

Two steps. **Energy:** if the loudest frame is under 10 dB, do nothing. **Templates:** cut out the loud part, stretch it to 32 frames, and find the nearest **template**: the average shape of each command (and of "other") in the training clips. The score is how much closer the clip is to the command's template than to the "other" template.

> **Predict.** With no threshold, how many of the validation commands does this baseline get right? And how many false alarms per hour?

In [ ]:
GATE_DB = 10.0       # quieter than this: no speech, the device does nothing

def frame_energy(db):
    """The loudness of each 10 ms frame: all mel bands together, in dB."""
    return 10 * np.log10((10 ** (db / 10)).sum(axis=0))

def shape_of(db):
    """The loud part of the clip, stretched to 32 frames, minus each band's mean: a 1,280-number shape."""
    e = frame_energy(db)
    if e.max() < GATE_DB:
        return None
    loud = np.where(e > e.max() - 15)[0]
    part = db[:, loud[0]:loud[-1] + 1]
    part = part[:, np.linspace(0, part.shape[1] - 1, 32).round().astype(int)]
    return (part - part.mean(axis=1, keepdims=True)).ravel()

shapes = [shape_of(from_uint8(p)) for p in pictures]
templates = {}
for c in COMMANDS + ["other"]:            # the mean shape of each word in the training clips
    templates[c] = np.mean([s for s, s_split, s_label in zip(shapes, split, label)
                            if s is not None and s_split == "train" and s_label == c], axis=0)

def template_heard(s):
    if s is None:
        return "", 0.0
    dist = {c: np.linalg.norm(s - t) for c, t in templates.items()}
    best = min(COMMANDS, key=dist.get)
    return best, float(dist["other"] - dist[best])   # score: how much closer to the command than to "other"

heard_template, score_template = map(np.array, zip(*[template_heard(s) for s in shapes]))
print(summary(evaluate(heard_template, score_template, threshold=0.0, part="valid")))

> **Check.** About 50.2% right, but about 3456.1 false alarms per hour: one average shape per word cannot hold thousands of voices, and loud noise passes the energy check.

## 6. A small CNN

A **filter** is a set of weights that slides over the input and gives a high number where its pattern is. Here each filter covers all 40 bands and 5 frames (50 ms) and slides along time. Four layers of 48 filters, then the strongest response of each filter, then 12 class probabilities. The next cell builds the network.

In [ ]:
import torch
import torch.nn as nn
torch.manual_seed(0)
torch.set_num_threads(2)

def standardise(q):
    """Each picture relative to its own loudest point, so a quiet and a loud voice look alike."""
    db = from_uint8(q)
    return np.maximum(db - db.max(axis=(-2, -1), keepdims=True), -80) / 20 + 2

def block(n_in, n_out):
    # 48 filters, each 40 bands x 5 frames (50 ms), slid along time; then keep the larger of each 2 frames
    return [nn.Conv1d(n_in, n_out, kernel_size=5, padding=2, bias=False), nn.BatchNorm1d(n_out), nn.ReLU(), nn.MaxPool1d(2)]

cnn = nn.Sequential(*block(40, 48), *block(48, 48), *block(48, 48), *block(48, 48),
                    nn.AdaptiveMaxPool1d(1), nn.Flatten(), nn.Dropout(0.2), nn.Linear(48, len(CLASSES)))
X = torch.tensor(standardise(pictures), dtype=torch.float32)   # items x 40 bands x 98 frames
Y = torch.tensor(y)
print("weights to learn:", sum(p.numel() for p in cnn.parameters()))

> **Check.** 45,132 weights to learn: about 0.18 MB as 32-bit numbers.

The next cell trains it: 12 epochs, batches of 64, each word moved in time by up to 60 ms at random. It takes about half a minute on a laptop (longer on a free Colab or Kaggle CPU).

In [ ]:
EPOCHS, BATCH = 12, 64
train_idx = np.where(split == "train")[0]
optimiser = torch.optim.Adam(cnn.parameters(), lr=4e-3)
schedule = torch.optim.lr_scheduler.OneCycleLR(optimiser, 4e-3, total_steps=EPOCHS * -(-len(train_idx) // BATCH))
rng = torch.Generator().manual_seed(0)
start = time.perf_counter()
for epoch in range(EPOCHS):
    cnn.train()
    order = train_idx[torch.randperm(len(train_idx), generator=rng).numpy()]
    for k in range(0, len(order), BATCH):
        batch = order[k:k + BATCH]
        shift = int(torch.randint(-6, 7, (1,), generator=rng))     # move the word up to 60 ms in time
        loss = nn.functional.cross_entropy(cnn(torch.roll(X[batch], shift, dims=2)), Y[batch])
        optimiser.zero_grad(); loss.backward(); optimiser.step(); schedule.step()
print(f"trained in {time.perf_counter() - start:.0f} s")

cnn.eval()
with torch.no_grad():                                     # the 12 class probabilities of every item
    probs = np.concatenate([torch.softmax(cnn(X[k:k + 2000]), dim=1).numpy() for k in range(0, len(X), 2000)])
best = probs.argmax(axis=1)
heard_cnn = np.array([CLASSES[b] if b < 10 else "" for b in best])
score_cnn = probs.max(axis=1)
print(summary(evaluate(heard_cnn, score_cnn, threshold=0.0, part="valid")))

> **Check.** On validation, with no threshold: about 90.2% right and about 203.1 false alarms per hour, none of them on noise. With another PyTorch version, your numbers can differ by a point or two.

## 7. Whisper tiny and Vosk (recorded)

The course team ran Whisper tiny (with the ten commands as its prompt) and Vosk small (with a grammar of the ten commands and `[unk]`) once on every validation and test item, and recorded the outputs. For Whisper, the word heard is the first word after the course's normalisation guide, if it is a command; the score is its average word probability. For Vosk, the score is its confidence in the word.

In [ ]:
def first_command(text):
    """The first word after the normalisation guide, if it is a command; otherwise none ("")."""
    words = cp.normalise(text, "full", "en").split()
    return words[0] if words and words[0] in COMMANDS else ""

recorded = {}
for name in ["whisper-tiny", "vosk-small"]:
    rows = [json.loads(line) for line in open(f"data/recordings/{name}.jsonl", encoding="utf-8")]
    recorded[name] = {r["id"] + "|" + r["split"]: r for r in rows}
key = [c["id"] + "|" + c["split"] for c in clips]

heard_whisper, score_whisper, heard_vosk, score_vosk = [], [], [], []
for k in key:
    w, v = recorded["whisper-tiny"].get(k), recorded["vosk-small"].get(k)
    heard_whisper.append(first_command(w["text"]) if w else "")
    score_whisper.append(float(np.exp(w["avg_logprob"])) if w else 0.0)     # Whisper's mean word probability
    vw = [x for x in v["words"] if x["word"] in COMMANDS] if v else []
    heard_vosk.append(vw[0]["word"] if vw else "")
    score_vosk.append(vw[0]["conf"] if vw else 0.0)                          # Vosk's confidence in that word
heard_whisper, score_whisper = np.array(heard_whisper), np.array(score_whisper)
heard_vosk, score_vosk = np.array(heard_vosk), np.array(score_vosk)
for name, h, s in [("whisper-tiny", heard_whisper, score_whisper), ("vosk-small", heard_vosk, score_vosk)]:
    print(name, summary(evaluate(h, s, threshold=0.0, part="valid")))

> **Check.** With no threshold: Whisper tiny about 91.2% right at 50.8 false alarms per hour; Vosk about 95.2% right at 344.2.

What did they write? A few validation clips:

In [ ]:
for k in ["learn/", "off/", "up/"]:
    rows = [(i, c) for i, c in enumerate(clips) if c["split"] == "valid" and c["id"].startswith(k)][:3]
    for i, c in rows:
        w = recorded["whisper-tiny"][key[i]]["text"]; v = recorded["vosk-small"][key[i]]["text"]
        print(f"{c['id']:28} Whisper: {w!r:24} Vosk: {v!r}")

## 8. One budget, one threshold rule

The four methods in one place, then the rule: the lowest threshold whose false alarms per hour on **validation** are within the budget.

> **Predict.** Which method keeps the most commands at 10 false alarms per hour? Can every method keep the budget?

In [ ]:
methods = {"energy + template": (heard_template, score_template), "small CNN": (heard_cnn, score_cnn),
           "Whisper tiny + word list": (heard_whisper, score_whisper), "Vosk + grammar": (heard_vosk, score_vosk)}

In [ ]:
BUDGET = 10   # false alarms per hour of other words and noise, on validation

def choose_threshold(heard, score, budget=BUDGET):
    """The lowest threshold whose false alarms per hour on validation are within the budget.
    If no threshold is, the strictest one (the method's highest score)."""
    candidates = np.unique(score[(split == "valid") & (heard != "")])
    for t in np.r_[0.0, candidates]:
        if evaluate(heard, score, t, "valid")["false alarms per hour"] <= budget:
            return float(t)
    return float(candidates.max())

thresholds = {name: choose_threshold(h, s) for name, (h, s) in methods.items()}
for name, (h, s) in methods.items():
    r = evaluate(h, s, thresholds[name], "valid")
    print(f"{name:26} threshold {thresholds[name]:7.3f}   right {r['right']:.3f}   false alarms per hour {r['false alarms per hour']}")

> **Check.** The CNN's threshold is about 0.947 and Whisper's about 0.613. Vosk cannot keep the budget: most of its answers have a confidence of exactly 1.0, so its strictest threshold is 1.0 and it still makes over 100 false alarms an hour.

## 9. The final test, once

The test split has not been used for any choice. Run each method once, at its validation threshold.

In [ ]:
table = []
for name, (h, s) in methods.items():
    r = evaluate(h, s, thresholds[name], "test")      # the test split, once, at the validation threshold
    table.append({"method": name, **summary(r)})
    print(f"{name:26} right {r['right']:.3f}  wrong word {r['wrong word']:.3f}  missed {r['missed']:.3f}  "
          f"false alarms/hour: other {r['per hour (other)']}, noise {r['per hour (noise)']}, all {r['false alarms per hour']}")

> **Check.** Small CNN about 67.7% right at 13.4 false alarms per hour; Whisper tiny 76.0% at 9.6; Vosk 93.7% at 145.8 (over the budget); the template 3.7%.

## 10. The worst cases: words and speakers

Commands caught per word, on test:

In [ ]:
per_word = {name: evaluate(h, s, thresholds[name], "test")["per_word"] for name, (h, s) in methods.items()}
print(f"{'word':6}" + "".join(f"{n[:14]:>16}" for n in per_word))
for w in COMMANDS:
    print(f"{w:6}" + "".join(f"{per_word[n][w]:16.2f}" for n in per_word))

Each method has its own weak words. Whisper tiny writes "on" correctly most of the time, but with a low probability, so one threshold cuts it.

And per speaker (test speakers with at least three command clips): the median, the mean of the worst tenth, and the worst.

In [ ]:
def speaker_worst(heard, score, threshold, min_clips=3):
    """Right answers per test speaker with at least `min_clips` command clips: the worst speakers."""
    acts = (heard != "") & (score >= threshold)
    ok = acts & (heard == label)
    rates = {}
    for p in set(speaker[(split == "test") & (y < 10)]):
        k = (split == "test") & (y < 10) & (speaker == p)
        if k.sum() >= min_clips:
            rates[p] = float(ok[k].mean())
    values = np.sort(list(rates.values()))
    tenth = max(1, len(values) // 10)
    return {"speakers": len(values), "median": round(float(np.median(values)), 3),
            "worst tenth": round(float(values[:tenth].mean()), 3), "worst": round(float(values[0]), 3)}

for name, (h, s) in methods.items():
    print(f"{name:26}", speaker_worst(h, s, thresholds[name]))

> **Warning.** The dataset has no age, gender or accent, so "the worst tenth of speakers" cannot tell you who they are. A pilot with real users must ask.

## 11. Speed and size

The course measured each method's time for one decision on one CPU thread of its laptop (`latency.json`). The next cell also times **your** CNN.

In [ ]:
latency = json.load(open("data/recordings/latency.json"))
x16 = [sf.read(f"data/audio/{w}")[0] for w in ["yes__43fc47a7_nohash_0.wav", "stop__0c40e715_nohash_0.wav"]]
torch.set_num_threads(1)
times = []
with torch.no_grad():
    for _ in range(20):
        for x in x16:
            t = time.perf_counter()
            cnn(torch.tensor(standardise(to_uint8(log_mel(x)))[None], dtype=torch.float32)).argmax()
            times.append(time.perf_counter() - t)
torch.set_num_threads(2)
print(f"your CNN: {1000 * np.median(times):.1f} ms a decision (picture included), one thread")
for name, k in [("energy + template", "template"), ("small CNN", "cnn"), ("Whisper tiny", "whisper"), ("Vosk small", "vosk")]:
    print(f"{name:18} {latency[k]['median_ms']:8.1f} ms a decision   {latency[k]['bytes'] / 1e6:8.2f} MB")

> **Check.** The CNN and the template take well under a millisecond and less than 0.2 MB; Vosk about 4 ms and 71 MB; Whisper tiny about 276 ms and 78 MB. The device decides every 100 ms, so Whisper cannot keep up even on a laptop core.

## 12. Failure case: no way to say "not a command"

A common mistake is to train or use the spotter as if every second held one of the ten commands.

In [ ]:
# The mistake: the CNN must answer with one of the ten commands (no "other", no "noise").
forced = np.array([COMMANDS[i] for i in probs[:, :10].argmax(axis=1)])
forced_result = evaluate(forced, np.ones(len(forced)), threshold=0.0, part="test")
print("right on commands:", forced_result["right"], "  false alarms per hour:", forced_result["false alarms per hour"])

> **Check.** About 93.7% right, which looks excellent, and about 3745.8 false alarms per hour: it acts on every other word and every noise window. **Fix:** give the model an "other" class and a "noise" class, and always report false alarms per hour next to the commands caught.

## 13. Change one thing: a stricter budget

> **Predict.** At 3 false alarms per hour instead of 10, how many commands does the CNN still catch on test?

In [ ]:
strict = {name: choose_threshold(h, s, budget=3) for name, (h, s) in methods.items()}
for name, (h, s) in methods.items():
    r = evaluate(h, s, strict[name], "test")
    print(f"{name:26} threshold {strict[name]:7.3f}   right {r['right']:.3f}   false alarms per hour {r['false alarms per hour']}")

> **Check.** About 59.1% for the CNN. Every step down in the budget costs commands, and with 21 minutes of validation negatives, the budget cannot go much lower: one false alarm is already 2.8 an hour.

## 14. End to end: the kitchen recording

A new recording: the test part of the dishes noise, with six test clips of new speakers placed in it (yes, bed, stop, go, seven, off). The CNN looks at the last second every 100 ms, acts when its score is at least its threshold, then waits one second.

In [ ]:
x, sr = sf.read("data/audio/kitchen-stream.wav")
truth = json.load(open("data/stream.json"))["words"]
STEP = 0.1                                                   # look at the last second, every 100 ms
starts = np.arange(0, len(x) / sr - 1 + 1e-9, STEP)
windows = np.stack([to_uint8(log_mel(x[int(t * sr):int(t * sr) + sr])) for t in starts])
with torch.no_grad():
    p = torch.softmax(cnn(torch.tensor(standardise(windows), dtype=torch.float32)), dim=1).numpy()
threshold = thresholds["small CNN"]
events, last = [], -9.0
for t, row in zip(starts, p):
    b = row.argmax()
    if b < 10 and row[b] >= threshold and t - last >= 1.0:     # act once, then wait 1 s
        events.append((round(float(t), 1), CLASSES[b], round(float(row[b]), 3)))
        last = t
print("said:  ", [(w["start"], w["word"]) for w in truth])
print("acted: ", events)
display(Audio(x, rate=sr))

> **Check.** The CNN acted on: yes at 1.7 s, stop at 7.5 s. It ignored the other words and the noise. The training pictures had speech **or** noise, never both: mixing training words with noise is the next change to try.

### Optional: run Whisper tiny and Vosk live

This cell downloads Whisper tiny (75 MB) and Vosk small (40 MB, checked against its SHA-256), and runs both on six test clips. It takes about a minute. Skip it if your connection is slow: the recorded outputs above are enough.

In [ ]:
# Optional: run both models live on six test clips (downloads Whisper tiny, 75 MB, and Vosk small, 40 MB).
import zipfile
from faster_whisper import WhisperModel
from huggingface_hub import snapshot_download
from vosk import Model, KaldiRecognizer, SetLogLevel
whisper = WhisperModel(snapshot_download("Systran/faster-whisper-tiny", revision="d90ca5fe260221311c53c58e660288d3deb8d356"),
                       device="cpu", compute_type="int8", cpu_threads=2)
if not Path("models/vosk-model-small-en-us-0.15").exists():
    Path("models").mkdir(exist_ok=True)
    download("https://alphacephei.com/vosk/models/vosk-model-small-en-us-0.15.zip", "models/vosk.zip")
    if sha256("models/vosk.zip") != "30f26242c4eb449f948e42cb302dd7a686cb29a3423a8367f99ff41780942498":
        raise SystemExit("The Vosk model has the wrong checksum. Delete the folder models and run again.")
    zipfile.ZipFile("models/vosk.zip").extractall("models")
SetLogLevel(-1)
vosk = Model("models/vosk-model-small-en-us-0.15")
for name in ["yes__43fc47a7_nohash_0.wav", "left__4c6167ca_nohash_6.wav", "off__e0c782d5_nohash_1.wav",
             "go__68dd409e_nohash_0.wav", "bed__1b4c9b89_nohash_0.wav", "three__8769c34c_nohash_2.wav"]:
    x, sr = sf.read(f"data/audio/{name}", dtype="float32")
    segments, _ = whisper.transcribe(x, language="en", beam_size=1, initial_prompt="yes, no, up, down, left, right, on, off, stop, go.",
                                     condition_on_previous_text=False, without_timestamps=True)
    rec = KaldiRecognizer(vosk, sr, json.dumps(COMMANDS + ["[unk]"]))
    rec.AcceptWaveform((x * 32767).astype("<i2").tobytes())
    print(f"{name.split('__')[0]:6} Whisper: {' '.join(s.text.strip() for s in segments)!r:22} Vosk: {json.loads(rec.FinalResult())['text']!r}")

> **Check.** Both should write "yes", "left", "off" and "go" for the first four clips. On "bed" and "three", Whisper writes the word itself, and Vosk writes `[unk]` or a command: its grammar has nothing else to say.

## 15. Your turn

> **Your turn.** (a) On the **test** other-word clips, how many times does Whisper tiny act at its threshold? Put the number in `task_whisper_other`.

In [ ]:
task_whisper_other = ...

In [ ]:
expect('task_whisper_other', task_whisper_other, 5)

> **Your turn.** (b) Choose the CNN's threshold for a budget of 20 false alarms per hour on validation (`choose_threshold(..., budget=20)`), then put its **right** share on **test** in `task_right`.

In [ ]:
task_threshold = ...
task_right = ...

In [ ]:
expect('task_right', task_right, 0.71, tolerance=0.05)

> **Your turn.** (c) Which command does the CNN catch least often on test, at its threshold? Put the word in `task_worst_word`.

In [ ]:
task_worst_word = ...

In [ ]:
expect_hash('task_worst_word', task_worst_word, 'cdbca2a1e5e496bb')

> **Check.** (b) has a tolerance, because the CNN's training differs a little between computers and PyTorch versions. If (c) says "Not yet" with the right method, look at your per-word table: with another PyTorch version, two weak words can swap places.

## Next

You built four keyword spotters, chose thresholds for a false-alarm budget on validation, tested once, and ran the winner on a new recording. The [case study page](https://learning.nextia-ai.com/courses/speech/m10/spot-commands-on-a-small-device/) has the recommendation, the full-data run, the costs, responsible use and how to make this a portfolio piece.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [An IVR for spoken intents](https://learning.nextia-ai.com/courses/speech/m10/an-ivr-for-spoken-intents/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/speech/m10/spot-commands-on-a-small-device/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The clips are from Speech Commands v0.02 (Pete Warden, Google, CC BY 4.0): real volunteers who recorded single words for this purpose.